# 🚀 CodeSage — AI-Powered Code Review (with CodeBERT)

This notebook extends the previous prototype by integrating a **real transformer model**
(`microsoft/codebert-base`) for contextual understanding of source code.

**Pipeline:**
1. Capture code
2. Static analysis (Pylint, Bandit, Radon, AST)
3. AI model analysis (CodeBERT / CodeT5)
4. Fusion + confidence scoring
5. Feedback visualization


In [19]:
import torch
import sys
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import ast

In [29]:
## retrain 
import torch
import pandas as pd
import json
import sys
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments

# --- 1. Configuration ---
MODEL_NAME = "Salesforce/codet5-base"
DATA_FILE = "generative_dataset.jsonl"
OUTPUT_DIR = "./trained-generative-model"  # <-- Saving to a new, clean directory
TRAIN_PREFIX = "suggest bug: "
NUM_EPOCHS = 15  # 15-20 epochs is good for a small dataset
BATCH_SIZE = 4

print(f"🚀 Starting generative model fine-tuning...")
print(f"   Base Model: {MODEL_NAME}")
print(f"   Data File:  {DATA_FILE}")
print(f"   Output Dir: {OUTPUT_DIR}\n")

# --- 2. Load Model and Tokenizer ---
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
except ImportError:
    print("--- ERROR: 'sentencepiece' not found. ---")
    print("Please install it: pip install sentencepiece")
    sys.exit(1)
except Exception as e:
    print(f"Error loading base model: {e}")
    sys.exit(1)
    
print("✅ Base model and tokenizer loaded.")

# --- 3. Prepare the Dataset ---
try:
    df = pd.read_json(DATA_FILE, lines=True)
    if not all(col in df.columns for col in ['code', 'suggestion']):
        print(f"Error: {DATA_FILE} must have 'code' and 'suggestion' columns.")
        sys.exit(1)
        
    print(f"✅ Loaded {len(df)} examples from {DATA_FILE}.")

except FileNotFoundError:
    print(f"--- ERROR: Data file not found! ---")
    print(f"Please make sure '{DATA_FILE}' is in the same directory.")
    sys.exit(1)
except Exception as e:
    print(f"Error reading data file: {e}")
    sys.exit(1)

# We add the prefix so the model knows what we're asking
inputs = [TRAIN_PREFIX + code for code in df["code"]]
targets = df["suggestion"].tolist()

# Tokenize the data
model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length").input_ids

# The model needs the tokenized targets as 'labels'
model_inputs["labels"] = labels

class GenerativeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        # Return a dictionary of tensors
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

dataset = GenerativeDataset(model_inputs)
print("✅ Dataset prepared and tokenized.")

# --- 4. Set up Training Arguments ---
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,  # Not used here, but good practice
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=5,  # Log progress more often
    predict_with_generate=True, # Important for Seq2Seq models
)

# --- 5. Create and run the Trainer ---
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    # We don't have a validation set, but for a real project you would
    # eval_dataset=... 
)

print("\n--- 🚂 Starting Training ---")
try:
    trainer.train()
    print("--- ✅ Fine-tuning complete. ---")

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("\n--- 🔴 FATAL ERROR: CUDA out of memory! ---")
        print("Your GPU doesn't have enough memory.")
        print("Try reducing BATCH_SIZE at the top of the script (e.g., to 2 or 1).")
        sys.exit(1)
    else:
        raise e

# --- 6. Save the final model ---
print(f"💾 Saving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Generative model saved successfully!")

🚀 Starting generative model fine-tuning...
   Base Model: Salesforce/codet5-base
   Data File:  generative_dataset.jsonl
   Output Dir: ./trained-generative-model

✅ Base model and tokenizer loaded.
✅ Loaded 10 examples from generative_dataset.jsonl.
✅ Dataset prepared and tokenized.

--- 🚂 Starting Training ---


d:\Final Year Project\CodeSage\venv\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,6.132700
10,4.156300
15,2.252300
20,1.817500
25,1.410600
30,1.175800
35,0.977000
40,0.966400
45,0.822700


--- ✅ Fine-tuning complete. ---
💾 Saving model to ./trained-generative-model...


SafetensorError: Error while serializing: I/O error: The requested operation cannot be performed on a file with a user-mapped section open. (os error 1224)

In [21]:
AI_MODEL_PATH = "./trained-generative-model"
AI_PREFIX = "suggest bug: "
MAX_INPUT_LENGTH = 512
MAX_OUTPUT_LENGTH = 128

In [22]:
def get_ai_suggestion(code_block, tokenizer, model, function_name=""):
    """
    Gets suggestions from trained model with better preprocessing.
    """
    try:
        # Normalize whitespace (convert tabs to spaces, normalize indents)
        normalized_code = code_block.replace('\t', '    ')
        
        # Create a more detailed prompt
        prompt = f"{AI_PREFIX}{normalized_code}"
        
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding=True
        )

        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids,
                max_length=MAX_OUTPUT_LENGTH,
                num_beams=5,
                temperature=0.7,
                early_stopping=True,
                no_repeat_ngram_size=3,
                top_p=0.9
            )
        
        suggestion = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        
        # Filter out non-suggestions
        if suggestion and len(suggestion) > 10 and suggestion != normalized_code:
            return suggestion
        return None
        
    except Exception as e:
        print(f"  ⚠️ AI model error: {e}")
        return None

In [23]:
def extract_functions(code_content):
    """
    Extract individual functions from code for targeted analysis.
    """
    functions = []
    try:
        tree = ast.parse(code_content)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                # Get the source code of the function
                func_lines = code_content.split('\n')[node.lineno - 1:node.end_lineno]
                func_code = '\n'.join(func_lines)
                functions.append({
                    'name': node.name,
                    'code': func_code,
                    'lineno': node.lineno
                })
    except Exception as e:
        print(f"  ⚠️ Function extraction error: {e}")
    
    return functions


In [24]:
sample_code = """
def buggy_divide(a, b):
    return a / b

def process(data):
    for i in range(len(data)):
        if data[i] == 0:
            print("Zero found")
    return sum(data)

def unsafe_operation():
    try:
        risky_function()
    except:
        pass
"""

file_name = "temp_code.py"
try:
    with open(file_name, "w") as f:
        f.write(sample_code.strip())
    print(f"✅ Created {file_name} for analysis")
except IOError as e:
    print(f"❌ Error: Could not write file {file_name}. {e}")
    sys.exit(1)

print("=" * 60)
print("🚀 CodeSage AI Code Reviewer - Enhanced Edition")
print("=" * 60)

# --- Load AI Model ---
print("\n--- 🧠 Loading AI Model ---")
ai_available = False
try:
    ai_tokenizer = AutoTokenizer.from_pretrained(AI_MODEL_PATH)
    ai_model = AutoModelForSeq2SeqLM.from_pretrained(AI_MODEL_PATH)
    ai_model.eval()  # Set to evaluation mode
    print("  ✅ AI model loaded successfully")
    ai_available = True
except Exception as e:
    print(f"  ⚠️ AI model not available: {e}")
    print("  → Falling back to rule-based analysis")

✅ Created temp_code.py for analysis
🚀 CodeSage AI Code Reviewer - Enhanced Edition

--- 🧠 Loading AI Model ---
  ✅ AI model loaded successfully


In [25]:
def analyze_code_patterns(code_content):
    """
    Rule-based analysis for common bugs when AI fails.
    """
    issues = []
    
    try:
        tree = ast.parse(code_content)
        
        for node in ast.walk(tree):
            # Check for division without ZeroDivisionError handling
            if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div):
                issues.append({
                    'line': node.lineno,
                    'type': 'potential_zero_division',
                    'message': 'Division operation without zero check - potential ZeroDivisionError',
                    'severity': 'high'
                })
            
            # Check for bare except clauses
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                issues.append({
                    'line': node.lineno,
                    'type': 'bare_except',
                    'message': 'Bare except clause catches all exceptions including system exits',
                    'severity': 'medium'
                })
            
            # Check for range(len()) pattern
            if isinstance(node, ast.Call):
                if (isinstance(node.func, ast.Name) and node.func.id == 'range' and
                    len(node.args) == 1 and isinstance(node.args[0], ast.Call)):
                    inner_call = node.args[0]
                    if isinstance(inner_call.func, ast.Name) and inner_call.func.id == 'len':
                        issues.append({
                            'line': node.lineno,
                            'type': 'range_len_pattern',
                            'message': 'Use enumerate() instead of range(len()) for better readability',
                            'severity': 'low'
                        })
    
    except Exception as e:
        print(f"  ⚠️ AST analysis error: {e}")
    
    return issues

In [26]:
with open(file_name) as f:
    code_content = f.read()

In [27]:
# --- AI + Rule-Based Bug Detection ---
print("\n--- 🤖 AI-Powered Bug Detection ---")

# First, try rule-based analysis
print("  → Running pattern-based analysis...")
pattern_issues = analyze_code_patterns(code_content)

if pattern_issues:
    for issue in pattern_issues:
        severity_icon = "🔴" if issue['severity'] == 'high' else "🟡" if issue['severity'] == 'medium' else "🔵"
        print(f"  {severity_icon} Line {issue['line']}: {issue['message']}")

# Then try AI analysis on each function
if ai_available:
    print("\n  → Running AI analysis on functions...")
    functions = extract_functions(code_content)
    
    if not functions:
        print("  ⚠️ No functions found to analyze")
    else:
        for func in functions:
            print(f"\n  Analyzing '{func['name']}'...")
            suggestion = get_ai_suggestion(func['code'], ai_tokenizer, ai_model, func['name'])
            
            if suggestion:
                print(f"    💡 AI Suggestion: {suggestion}")
            else:
                print(f"    ℹ️ No AI suggestions (function may be correct)")
else:
    print("\n  ℹ️ AI analysis skipped (model not available)")

# --- Summary ---
print("\n" + "=" * 60)
print("📊 Analysis Summary")
print("=" * 60)
print(f"  • Functions analyzed: {len(functions) if 'functions' in locals() else 0}")
print(f"  • Pattern-based issues: {len(pattern_issues)}")
print(f"  • AI model: {'Active' if ai_available else 'Inactive (using rules)'}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- 🤖 AI-Powered Bug Detection ---
  → Running pattern-based analysis...
  🔴 Line 2: Division operation without zero check - potential ZeroDivisionError
  🔵 Line 5: Use enumerate() instead of range(len()) for better readability
  🟡 Line 13: Bare except clause catches all exceptions including system exits

  → Running AI analysis on functions...

  Analyzing 'buggy_divide'...
    💡 AI Suggestion: 'b' is a special type for rounding errors. Consider adding a try-except block to handle the ZeroDivisionError.

  Analyzing 'process'...
    💡 AI Suggestion: a function like 'sum' can cause a ZeroDivisionError. Consider adding a try-except block to handle zeroes.

  Analyzing 'unsafe_operation'...
    💡 AI Suggestion: 'unsafe_operation()' is a bug and can lead to unexpected behavior. Consider adding a try-except block to catch exceptions like 'ValueError' or 'IOError'.

📊 Analysis Summary
  • Functions analyzed: 3
  • Pattern-based issues: 3
  • AI model: Active
